# CSVLoader vs. UnstructuredExcelLoader in LangChain

When ingesting tabular data into Retrieval-Augmented Generation (RAG) pipelines, choosing the right loader depends on whether your data comes in flat, structured CSV files or complex, multi-sheet Excel workbooks. Below is a detailed feature-by-feature comparison, followed by the pros and cons of each.

---

## Comparative Matrix

| Feature / Metric | `CSVLoader` | `UnstructuredExcelLoader` |
| :--- | :--- | :--- |
| **Primary File Format** | `.csv` (Comma-Separated Values) | `.xlsx`, `.xls` (Microsoft Excel) |
| **LangChain Import Class** | `from langchain_community.document_loaders import CSVLoader` | `from langchain_community.document_loaders import UnstructuredExcelLoader` |
| **Underlying Engine** | Python's native `csv` module | `unstructured` + `pandas` / `openpyxl` / `msoffcrypto-tool` |
| **Document Granularity** | **1 Document per row** | **1 Document total** (`mode="single"`) OR **1 Document per element/table** (`mode="elements"`) |
| **Multi-Sheet Support** | ❌ No (CSVs are single-sheet flat text files) | ✅ Yes (Extracts and parses all sheets across a workbook) |
| **Speed & Performance** | **Extremely Fast & Lightweight** | **Slower** (heavier parsing overhead) |
| **Dependency Footprint** | Minimal / None (Standard Python library) | Heavy (`unstructured`, `pandas`, `openpyxl`, `msoffcrypto-tool`) |
| **Formatting & Metadata Handling** | Converts each key-value pair into key:value text format; lightweight metadata (source, row index) | Extracts HTML table formatting, cell coordinates, sheet metadata, and element categories |

---

## Detailed Overview

### 1. CSVLoader

`CSVLoader` is designed for simple, row-by-row tabular text processing. It parses flat text files and outputs a distinct `Document` object for every row, converting column headers into text key-value pairs.

#### Pros
* **Lightweight & Fast:** High parsing speed with minimal computational overhead.
* **Zero Complex Dependencies:** Works out of the box without requiring large external libraries.
* **Row-Level Precision:** Ideal for row-based vector store indexing, allowing granular retrieval of specific entries.
* **Custom Field Mapping:** Allows selecting specific column fields as source metadata or content filters.

#### Cons
* **Format Limitation:** Cannot natively read Excel `.xlsx` or `.xls` files without prior conversion to `.csv`.
* **No Multi-Sheet Context:** Cannot parse multi-tab structures or cross-table relationships natively.
* **Flat Text Output:** Loses formatting context when handling complex nested headers or aggregated calculations.

---

### 2. UnstructuredExcelLoader

`UnstructuredExcelLoader` is designed for rich spreadsheet analysis. It uses layout and element parsing to extract single or multi-sheet Excel files, converting tabular data into structured HTML or text blocks.

#### Pros
* **Multi-Sheet Awareness:** Easily parses complex multi-tab workbooks in a single run.
* **Element-Level Extraction (`mode="elements"`):** Can separate titles, text summaries, and individual tables into distinct structural elements.
* **HTML Table Retention:** Retains standard HTML table structures (`<table>...</table>`), preserving column alignment for better LLM comprehension during RAG generation.
* **Encrypted File Compatibility:** Can parse encrypted Office workbooks using underlying integrations (`msoffcrypto-tool`).

#### Cons
* **Heavy Dependency Overhead:** Requires a large installation footprint (`unstructured`, `pandas`, `openpyxl`, `msoffcrypto-tool`).
* **Slower Processing Speeds:** Significantly slower on large files compared to simple row-by-row streaming.
* **Resource Intensive:** Consumes more CPU and memory, making it less ideal for serverless or ultra-lightweight deployments.

---

## Usage Summary

* **Use `CSVLoader` if:** You are working with standard, single-sheet flat files (e.g., product catalogs, user logs, dataset exports) where each row represents a standalone record.
* **Use `UnstructuredExcelLoader` if:** You have multi-sheet corporate workbooks containing formatted tables, financial summaries, headers, and sheet-level structures that need to be preserved for contextual retrieval.

# Lets create Sample CSV Data

In [ ]:
import os
import pandas as pd
from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import UnstructuredCSVLoader

In [8]:
os.makedirs("data/csv_files", exist_ok= True)

dataa = {
    'Product': ['Mobile', 'Mouse', 'Keyboard', 'Monitor', 'Projector'],
        'Category': ['Electronics', 'Accessories', 'Accessories', 'Electronics', 'Electronics'],
        'Price': [399.99, 29.99, 79.99, 299.99, 189.99],
        'Stock': [50, 20, 15, 35, 10],
        'Description': [
            'High-performance mobile with 112 pxl camera 4 gb RAM and 12GB ROM',
            'Wireless optical mouse with ergonomic design',
            'Mechanical keyboard with RGB backlighting',
            '27-inch 4K monitor with HDR support',
            '1080p webcam with noise cancellation'
        ]
}

df = pd.DataFrame(dataa)
df.to_csv("data/csv_files/storagesample.csv",index = False)

df


,Product,Category,Price,Stock,Description
0,Mobile,Electronics,399.99,50,High-performance mobile with 112 pxl camera 4 ...
1,Mouse,Accessories,29.99,20,Wireless optical mouse with ergonomic design
2,Keyboard,Accessories,79.99,15,Mechanical keyboard with RGB backlighting
3,Monitor,Electronics,299.99,35,27-inch 4K monitor with HDR support
4,Projector,Electronics,189.99,10,1080p webcam with noise cancellation


In [13]:
# Lets Save as Excel with multiple sheet, we are using context manager bruh!!
with pd.ExcelWriter('data/csv_files/inventory.xlsx') as writer:
    df.to_excel(writer, sheet_name='products',index = False)
     # Add another sheet
    summary_data = {
            'Category': ['Electronics', 'Accessories'],
            'Total_Items': [3, 2],
            'Total_Value': [1475, 785]
        }
    df2 = pd.DataFrame(summary_data)
    df2.to_excel(writer, sheet_name='Summary', index=False)

print("Created the csv and excel type files :)")

Created the csv and excel type files :)


# CSV Processing

In [15]:
# Method 1: CSVLoader - Each row becomes a document
print("1️⃣ CSVLoader - Row-based Documents")
csv_loader = CSVLoader(
    file_path='data/csv_files/storagesample.csv',
    encoding='utf-8',
    csv_args={
        'delimiter': ',',
        'quotechar': '"',
    },
)

csv_docs = csv_loader.load() # load to get the doc into list
print(csv_docs)
print(f"Loaded {len(csv_docs)} documents (one per row)")
print("\nFirst document:")
print(f"Content: {csv_docs[0].page_content}")
print(f"Metadata: {csv_docs[0].metadata}")

1️⃣ CSVLoader - Row-based Documents
[Document(metadata={'source': 'data/csv_files/storagesample.csv', 'row': 0}, page_content='Product: Mobile\nCategory: Electronics\nPrice: 399.99\nStock: 50\nDescription: High-performance mobile with 112 pxl camera 4 gb RAM and 12GB ROM'), Document(metadata={'source': 'data/csv_files/storagesample.csv', 'row': 1}, page_content='Product: Mouse\nCategory: Accessories\nPrice: 29.99\nStock: 20\nDescription: Wireless optical mouse with ergonomic design'), Document(metadata={'source': 'data/csv_files/storagesample.csv', 'row': 2}, page_content='Product: Keyboard\nCategory: Accessories\nPrice: 79.99\nStock: 15\nDescription: Mechanical keyboard with RGB backlighting'), Document(metadata={'source': 'data/csv_files/storagesample.csv', 'row': 3}, page_content='Product: Monitor\nCategory: Electronics\nPrice: 299.99\nStock: 35\nDescription: 27-inch 4K monitor with HDR support'), Document(metadata={'source': 'data/csv_files/storagesample.csv', 'row': 4}, page_conte

In [31]:
# Lets create our metadataenricher a customized CSV...

from typing import List,Dict
from langchain_core.documents import Document

def cust_CSVingest(filepath:str)->List[Document]:
    df = pd.read_csv(filepath)
    documents = []
    # Create structured content as string using f """ """
    for idx,row in df.iterrows():
        info = f""" Product Information:
        Name: {row['Product']}
        Category: {row['Category']}
        Price: ₹{row['Price']}
        Stock: {row['Stock']} units
        Description: {row['Description']}"""
        doc = Document(
            page_content= info,
            metadata={
                'source': filepath,
                'row_number': idx+1,
                'product_name': row['Product'],
                'category': row['Category'],
                'price': row['Price'],
                'data_type': 'product_info'
            }
        )
        documents.append(doc)
    return documents

In [ ]:
cust_CSVingest("data/csv_files/storagesample.csv")

[Document(metadata={'source': 'data/csv_files/storagesample.csv', 'row_number': 1, 'product_name': 'Mobile', 'category': 'Electronics', 'price': 399.99, 'data_type': 'product_info'}, page_content=' Product Information:\n        Name: Mobile\n        Category: Electronics\n        Price: ₹399.99\n        Stock: 50 units\n        Description: High-performance mobile with 112 pxl camera 4 gb RAM and 12GB ROM'),
 Document(metadata={'source': 'data/csv_files/storagesample.csv', 'row_number': 2, 'product_name': 'Mouse', 'category': 'Accessories', 'price': 29.99, 'data_type': 'product_info'}, page_content=' Product Information:\n        Name: Mouse\n        Category: Accessories\n        Price: ₹29.99\n        Stock: 20 units\n        Description: Wireless optical mouse with ergonomic design'),
 Document(metadata={'source': 'data/csv_files/storagesample.csv', 'row_number': 3, 'product_name': 'Keyboard', 'category': 'Accessories', 'price': 79.99, 'data_type': 'product_info'}, page_content=' Pr

In [33]:
# 📊 CSV Processing Strategies
print("\n📊 CSV Processing Strategies:")
print("\n1. Row-based (CSVLoader):")
print("  ✅ Simple one-row-one-document")
print("  ✅ Good for record lookups")
print("  ❌ Loses table context")

print("\n2. Customized Processing:")
print("  ✅ Preserves relationships")
print("  ✅ Creates summaries")
print("  ✅ Rich metadata")
print("  ✅ Better for Q&A")


📊 CSV Processing Strategies:

1. Row-based (CSVLoader):
  ✅ Simple one-row-one-document
  ✅ Good for record lookups
  ❌ Loses table context

2. Customized Processing:
  ✅ Preserves relationships
  ✅ Creates summaries
  ✅ Rich metadata
  ✅ Better for Q&A


# Excel Processing with UnstructuredExcelLoader and CUstomized Loader


In [39]:
from langchain_community.document_loaders import UnstructuredExcelLoader


try:
    excel_loader = UnstructuredExcelLoader(
        'data/csv_files/inventory.xlsx',
        mode="elements"
    )
    unstructured_docs = excel_loader.load()
    
except Exception as e:
    print(f'Uh oww error:',e)

In [43]:
unstructured_docs[0].metadata
unstructured_docs[1].metadata
unstructured_docs[0].page_content

'Product Category Price Stock Description Mobile Electronics 399.99 50 High-performance mobile with 112 pxl camera 4 gb RAM and 12GB ROM Mouse Accessories 29.99 20 Wireless optical mouse with ergonomic design Keyboard Accessories 79.99 15 Mechanical keyboard with RGB backlighting Monitor Electronics 299.99 35 27-inch 4K monitor with HDR support Projector Electronics 189.99 10 1080p webcam with noise cancellation'

In [44]:
def process_excel_with_pandas(filepath: str) -> List[Document]:
    """Process Excel with sheet awareness"""
    documents = []
    
    # Read all sheets
    excel_file = pd.ExcelFile(filepath) # for csv we use read_csv
    
    for sheet_name in excel_file.sheet_names:           # reading individual sheet names
        df = pd.read_excel(filepath, sheet_name=sheet_name)
        
        # Create document for each sheet
        sheet_content = f"Sheet: {sheet_name}\n"
        sheet_content += f"Columns: {', '.join(df.columns)}\n"
        sheet_content += f"Rows: {len(df)}\n\n"
        sheet_content += df.to_string(index=False)
        
        doc = Document(
            page_content=sheet_content,
            metadata={
                'source': filepath,
                'sheet_name': sheet_name,
                'num_rows': len(df),
                'num_columns': len(df.columns),
                'data_type': 'excel_sheet'
            }
        )
        documents.append(doc)
    
    return documents

In [53]:
excel_docs = process_excel_with_pandas('data/csv_files/inventory.xlsx')
print(f"Processed {len(excel_docs)} sheets")
# print(excel_docs[0].page_content)
print(excel_docs[1].metadata)


Processed 2 sheets
{'source': 'data/csv_files/inventory.xlsx', 'sheet_name': 'Summary', 'num_rows': 2, 'num_columns': 3, 'data_type': 'excel_sheet'}


In [54]:
excel_docs

[Document(metadata={'source': 'data/csv_files/inventory.xlsx', 'sheet_name': 'products', 'num_rows': 5, 'num_columns': 5, 'data_type': 'excel_sheet'}, page_content='Sheet: products\nColumns: Product, Category, Price, Stock, Description\nRows: 5\n\n  Product    Category  Price  Stock                                                       Description\n   Mobile Electronics 399.99     50 High-performance mobile with 112 pxl camera 4 gb RAM and 12GB ROM\n    Mouse Accessories  29.99     20                      Wireless optical mouse with ergonomic design\n Keyboard Accessories  79.99     15                         Mechanical keyboard with RGB backlighting\n  Monitor Electronics 299.99     35                               27-inch 4K monitor with HDR support\nProjector Electronics 189.99     10                              1080p webcam with noise cancellation'),
 Document(metadata={'source': 'data/csv_files/inventory.xlsx', 'sheet_name': 'Summary', 'num_rows': 2, 'num_columns': 3, 'data_type'